# ESS Battery Health — Batch 1·2·3 비교 EDA

- 데이터셋: Stanford & MIT, *Data-driven prediction of battery cycle life before capacity degradation*
- Batch 1: 학습 데이터
- Batch 2: 1차 테스트 데이터
- Batch 3: 2차 테스트 데이터
- 목적: 동일한 기준으로 세 배치를 탐색하고, 초기 100사이클 기반 수명 회귀 전략을 수립

> 과제 요구에 따라 테스트 배치도 EDA에서 비교합니다. 모델 학습과 하이퍼파라미터 선택은 Batch 1 내부 교차검증으로만 수행하고, Batch 2·3의 성능은 최종 평가에만 사용합니다.

In [ ]:
from pathlib import Path
import platform
import warnings

import h5py
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
EOL_CAPACITY_AH = 0.88

print(f'Python: {platform.python_version()}')
print(f'numpy: {np.__version__}')
print(f'pandas: {pd.__version__}')
print(f'h5py: {h5py.__version__}')
print(f'matplotlib: {matplotlib.__version__}')

In [ ]:
# 저장소 루트 또는 notebooks/ 디렉터리에서 실행해도 data/를 찾습니다.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'

BATCH_FILES = {
    'Batch 1': DATA_DIR / '2017-05-12_batchdata_updated_struct_errorcorrect.mat',
    'Batch 2': DATA_DIR / '2018-02-20_batchdata_updated_struct_errorcorrect.mat',
    'Batch 3': DATA_DIR / '2018-04-12_batchdata_updated_struct_errorcorrect.mat',
}

for batch_name, file_path in BATCH_FILES.items():
    if not file_path.exists():
        raise FileNotFoundError(f'{batch_name} 파일을 찾을 수 없습니다: {file_path}')

    size_gb = file_path.stat().st_size / (1024 ** 3)
    print(f'{batch_name}: {file_path.name} ({size_gb:.2f} GiB)')

## 0. 공통 데이터 로딩

대용량 MAT 파일을 통째로 메모리에 올리지 않고 필요한 HDF5 객체만 읽습니다. 요약 데이터와 10·100사이클의 `Qdlin`만 추출합니다.

In [ ]:
SUMMARY_FIELDS = (
    'cycle', 'QDischarge', 'QCharge', 'IR',
    'Tmax', 'Tavg', 'Tmin', 'chargetime',
)


def read_vector(hdf_object):
    """HDF5 데이터셋을 1차원 NumPy 배열로 읽습니다."""
    return np.asarray(hdf_object[()]).reshape(-1)


def dereference(mat_file, reference_dataset, index):
    """MATLAB struct 필드의 객체 참조를 실제 HDF5 객체로 변환합니다."""
    return mat_file[reference_dataset[index, 0]]


def read_scalar_reference(mat_file, reference_dataset, index):
    """객체 참조가 가리키는 첫 번째 스칼라 값을 읽습니다."""
    values = read_vector(dereference(mat_file, reference_dataset, index))
    return float(values[0]) if len(values) else np.nan


def read_text_reference(mat_file, reference_dataset, index):
    """MATLAB 문자 배열을 Python 문자열로 변환합니다."""
    values = read_vector(dereference(mat_file, reference_dataset, index))

    if np.issubdtype(values.dtype, np.integer):
        return ''.join(chr(int(value)) for value in values if int(value)).strip()

    return str(values[0]) if len(values) else 'unknown'


def load_batch(file_path, batch_name):
    """한 배치의 셀 정보, 사이클 요약, ΔQ(V) 데이터를 선택적으로 읽습니다."""
    cell_records = []
    summary_records = []
    delta_records = []

    with h5py.File(file_path, 'r') as mat_file:
        batch = mat_file['batch']
        cell_count = batch['cycle_life'].shape[0]

        for cell_id in range(cell_count):
            cell_key = f'{batch_name}-C{cell_id:02d}'
            cycle_life = read_scalar_reference(
                mat_file, batch['cycle_life'], cell_id
            )
            charging_policy = read_text_reference(
                mat_file, batch['policy_readable'], cell_id
            )

            cell_records.append({
                'batch': batch_name,
                'cell_id': cell_id,
                'cell_key': cell_key,
                'cycle_life': cycle_life,
                'charging_policy': charging_policy,
            })

            summary_group = dereference(mat_file, batch['summary'], cell_id)
            summary_arrays = {
                field: read_vector(summary_group[field]).astype(float)
                for field in SUMMARY_FIELDS
            }
            summary_length = min(len(values) for values in summary_arrays.values())

            for row_index in range(summary_length):
                summary_records.append({
                    'batch': batch_name,
                    'cell_id': cell_id,
                    'cell_key': cell_key,
                    'cycle_life': cycle_life,
                    'charging_policy': charging_policy,
                    **{
                        field: summary_arrays[field][row_index]
                        for field in SUMMARY_FIELDS
                    },
                })

            # 초기 100사이클이 존재하는 셀에서 ΔQ100-10(V)를 계산합니다.
            cycles_group = dereference(mat_file, batch['cycles'], cell_id)
            if cycles_group['Qdlin'].shape[0] < 100:
                continue

            voltage = read_vector(
                dereference(mat_file, batch['Vdlin'], cell_id)
            ).astype(float)
            qd_cycle_10 = read_vector(
                mat_file[cycles_group['Qdlin'][9, 0]]
            ).astype(float)
            qd_cycle_100 = read_vector(
                mat_file[cycles_group['Qdlin'][99, 0]]
            ).astype(float)

            common_length = min(
                len(voltage), len(qd_cycle_10), len(qd_cycle_100)
            )
            voltage = voltage[:common_length]
            delta_q = qd_cycle_100[:common_length] - qd_cycle_10[:common_length]
            valid = np.isfinite(voltage) & np.isfinite(delta_q)
            voltage = voltage[valid]
            delta_q = delta_q[valid]

            if len(delta_q) == 0:
                continue

            delta_variance = float(np.var(delta_q, ddof=1))
            delta_records.append({
                'batch': batch_name,
                'cell_id': cell_id,
                'cell_key': cell_key,
                'cycle_life': cycle_life,
                'voltage': voltage,
                'delta_q': delta_q,
                'delta_q_variance': delta_variance,
                'log_delta_q_variance': np.log10(
                    max(delta_variance, np.finfo(float).tiny)
                ),
                'delta_q_min': float(delta_q.min()),
                'delta_q_mean': float(delta_q.mean()),
            })

    return cell_records, summary_records, delta_records

In [ ]:
all_cells = []
all_summaries = []
all_deltas = []

for batch_name, file_path in BATCH_FILES.items():
    cells, summaries, deltas = load_batch(file_path, batch_name)
    all_cells.extend(cells)
    all_summaries.extend(summaries)
    all_deltas.extend(deltas)
    print(
        f'{batch_name} 로딩 완료: '
        f'{len(cells)} cells, {len(summaries):,} cycle rows'
    )

cell_df = pd.DataFrame(all_cells)
cycle_df = pd.DataFrame(all_summaries)
delta_df = pd.DataFrame(all_deltas)

# Target을 사용하는 분석에서는 cycle_life 결측 셀을 제외합니다.
valid_cell_df = cell_df.dropna(subset=['cycle_life']).copy()
valid_cycle_df = cycle_df.dropna(subset=['cycle_life']).copy()
valid_delta_df = delta_df.dropna(subset=['cycle_life']).copy()

print(f'전체 셀: {len(cell_df)}')
print(f'유효 Target 셀: {len(valid_cell_df)}')
print(f'사이클 요약 행: {len(cycle_df):,}')

In [ ]:
quality_table = (
    cell_df.groupby('batch', observed=True)
    .agg(
        total_cells=('cell_key', 'nunique'),
        valid_target_cells=('cycle_life', 'count'),
        missing_target_cells=('cycle_life', lambda values: values.isna().sum()),
        policy_count=('charging_policy', 'nunique'),
    )
)

display(quality_table)

missing_targets = cell_df[cell_df['cycle_life'].isna()][
    ['batch', 'cell_id', 'cell_key', 'charging_policy']
]
if not missing_targets.empty:
    print('cycle_life 결측 셀')
    display(missing_targets)

## 1. Cycle Life 분포는 어떻게 생겼는가?

동일한 bin과 축으로 세 배치의 수명 분포를 비교합니다.

In [ ]:
batch_order = list(BATCH_FILES.keys())
global_min = valid_cell_df['cycle_life'].min()
global_max = valid_cell_df['cycle_life'].max()
histogram_bins = np.linspace(global_min, global_max, 21)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

for axis, batch_name in zip(axes, batch_order):
    values = valid_cell_df.loc[
        valid_cell_df['batch'] == batch_name, 'cycle_life'
    ]
    axis.hist(values, bins=histogram_bins, color='steelblue', edgecolor='white')
    axis.axvline(values.mean(), color='tomato', linestyle='--',
                 label=f'Mean {values.mean():.1f}')
    axis.axvline(values.median(), color='darkgreen', linestyle=':',
                 label=f'Median {values.median():.1f}')
    axis.set_title(f'{batch_name} (n={len(values)})')
    axis.set_xlabel('Cycle Life')
    axis.legend(fontsize=9)

axes[0].set_ylabel('Number of Cells')
fig.suptitle('Cycle Life Distribution by Batch', fontweight='bold')
plt.tight_layout()
plt.show()

cycle_life_summary = (
    valid_cell_df.groupby('batch', observed=True)['cycle_life']
    .describe()
    .rename(columns={'50%': 'median'})
)
cycle_life_summary['over_1000_count'] = (
    valid_cell_df.assign(over_1000=valid_cell_df['cycle_life'] > 1000)
    .groupby('batch', observed=True)['over_1000'].sum()
)
cycle_life_summary['under_500_count'] = (
    valid_cell_df.assign(under_500=valid_cell_df['cycle_life'] < 500)
    .groupby('batch', observed=True)['under_500'].sum()
)
display(cycle_life_summary.round(2))

## 2. 열화 곡선 — 방전 용량은 어떻게 감소하는가?

명백한 측정 오류 범위만 제외하고, 동일한 축·색상 기준·EOL 기준선으로 세 배치를 비교합니다. 단수명은 빨간색, 장수명은 파란색입니다.

In [ ]:
clean_cycle_df = valid_cycle_df[
    valid_cycle_df['QDischarge'].between(0.5, 1.3)
].copy()

life_norm = plt.Normalize(
    valid_cell_df['cycle_life'].min(),
    valid_cell_df['cycle_life'].max(),
)
life_cmap = cm.coolwarm_r

fig, axes = plt.subplots(
    1, 3, figsize=(19, 6), sharex=True, sharey=True, constrained_layout=True
)

for axis, batch_name in zip(axes, batch_order):
    batch_cycles = clean_cycle_df[clean_cycle_df['batch'] == batch_name]

    for _, cell_data in batch_cycles.groupby('cell_key'):
        cycle_life = cell_data['cycle_life'].iloc[0]
        axis.plot(
            cell_data['cycle'],
            cell_data['QDischarge'],
            color=life_cmap(life_norm(cycle_life)),
            linewidth=0.7,
            alpha=0.65,
        )

    axis.axhline(EOL_CAPACITY_AH, color='black', linestyle='--',
                 linewidth=1.3, label='EOL (0.88 Ah)')
    axis.set_title(batch_name)
    axis.set_xlabel('Cycle')
    axis.set_ylim(0.8, 1.2)
    axis.legend(fontsize=9)

axes[0].set_ylabel('QDischarge (Ah)')
color_map = cm.ScalarMappable(norm=life_norm, cmap=life_cmap)
color_map.set_array([])
colorbar = fig.colorbar(color_map, ax=axes, pad=0.01)
colorbar.set_label('Cycle Life')
fig.suptitle('Discharge Capacity Degradation by Batch', fontweight='bold')
plt.show()

### 2-1. Knee point 및 열화 가속 비교

21개 관측치 이동 중앙값으로 평활화한 뒤, 두 선형 구간의 오차 합이 가장 작아지는 사이클을 Knee point로 추정합니다. 마지막 QD가 0.90Ah 이하인 곡선만 사용합니다. 이 값은 전체 수명 정보를 사용하므로 모델 피처가 아닌 EDA 전용입니다.

In [ ]:
SMOOTHING_WINDOW = 21
COMPLETE_CURVE_QD_LIMIT = 0.90
MIN_SEGMENT_RATIO = 0.15
MIN_SEGMENT_POINTS = 30


def fit_linear_segment(cycles, capacities):
    """한 구간의 선형 기울기와 잔차 제곱합을 반환합니다."""
    slope, intercept = np.polyfit(cycles, capacities, 1)
    predictions = slope * cycles + intercept
    sse = np.square(capacities - predictions).sum()
    return float(slope), float(sse)


def estimate_knee_point(cell_data):
    """두 선형 구간의 총 오차가 최소인 Knee point를 추정합니다."""
    ordered = cell_data.sort_values('cycle')
    cycles = ordered['cycle'].to_numpy(dtype=float)
    capacities = ordered['QDischarge'].to_numpy(dtype=float)
    smoothed = (
        pd.Series(capacities)
        .rolling(SMOOTHING_WINDOW, center=True, min_periods=1)
        .median()
        .to_numpy()
    )
    min_size = max(MIN_SEGMENT_POINTS, int(len(cycles) * MIN_SEGMENT_RATIO))

    if len(cycles) < min_size * 2:
        return None

    best = None
    for split in range(min_size, len(cycles) - min_size):
        before_slope, before_sse = fit_linear_segment(
            cycles[:split], smoothed[:split]
        )
        after_slope, after_sse = fit_linear_segment(
            cycles[split:], smoothed[split:]
        )
        candidate = {
            'total_sse': before_sse + after_sse,
            'split': split,
            'before_slope': before_slope,
            'after_slope': after_slope,
        }
        if best is None or candidate['total_sse'] < best['total_sse']:
            best = candidate

    return {
        'knee_cycle': cycles[best['split']],
        'before_slope': best['before_slope'],
        'after_slope': best['after_slope'],
    }


def build_knee_table(degradation_data):
    """세 배치의 셀별 Knee point 표를 생성합니다."""
    records = []

    for (batch_name, cell_key), cell_data in degradation_data.groupby(
        ['batch', 'cell_key'], observed=True
    ):
        ordered = cell_data.sort_values('cycle')
        final_qd = float(ordered['QDischarge'].iloc[-1])
        if final_qd > COMPLETE_CURVE_QD_LIMIT:
            continue

        knee = estimate_knee_point(ordered)
        if knee is None:
            continue

        cycle_life = float(ordered['cycle_life'].iloc[0])
        before_slope = knee['before_slope'] * 1000
        after_slope = knee['after_slope'] * 1000
        records.append({
            'batch': batch_name,
            'cell_key': cell_key,
            'cycle_life': cycle_life,
            'final_QD_Ah': final_qd,
            'knee_cycle': knee['knee_cycle'],
            'knee_life_ratio_pct': knee['knee_cycle'] / cycle_life * 100,
            'before_slope_mAh_per_cycle': before_slope,
            'after_slope_mAh_per_cycle': after_slope,
            'acceleration_ratio': (
                abs(after_slope) / max(abs(before_slope), 1e-12)
            ),
        })

    return pd.DataFrame(records)


knee_df = build_knee_table(clean_cycle_df)
knee_summary = (
    knee_df.groupby('batch', observed=True)
    .agg(
        analyzed_cells=('cell_key', 'nunique'),
        knee_mean=('knee_cycle', 'mean'),
        knee_median=('knee_cycle', 'median'),
        knee_ratio_mean_pct=('knee_life_ratio_pct', 'mean'),
        before_slope_mean=('before_slope_mAh_per_cycle', 'mean'),
        after_slope_mean=('after_slope_mAh_per_cycle', 'mean'),
        acceleration_mean=('acceleration_ratio', 'mean'),
    )
)
display(knee_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
knee_df.boxplot(column='knee_life_ratio_pct', by='batch', ax=axes[0])
knee_df.boxplot(column='acceleration_ratio', by='batch', ax=axes[1])
axes[0].set_title('Knee Position')
axes[0].set_ylabel('Knee / Cycle Life (%)')
axes[1].set_title('Degradation Acceleration')
axes[1].set_ylabel('After / Before Slope')
for axis in axes:
    axis.set_xlabel('Batch')
fig.suptitle('')
plt.tight_layout()
plt.show()

## 3. ΔQ100–10(V) — 초기 사이클에서 차이가 보이는가?

배치별 중앙 ΔQ 곡선과 IQR을 비교하고, 핵심 피처인 `log10(var(ΔQ))`와 Cycle Life의 관계를 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
batch_colors = {
    'Batch 1': 'steelblue',
    'Batch 2': 'tomato',
    'Batch 3': 'darkgreen',
}

for batch_name in batch_order:
    batch_delta = valid_delta_df[valid_delta_df['batch'] == batch_name]
    minimum_length = min(len(values) for values in batch_delta['delta_q'])
    delta_matrix = np.vstack([
        values[:minimum_length] for values in batch_delta['delta_q']
    ])
    voltage = batch_delta['voltage'].iloc[0][:minimum_length]
    order = np.argsort(voltage)
    voltage = voltage[order]
    delta_matrix = delta_matrix[:, order]

    median_curve = np.median(delta_matrix, axis=0)
    lower_curve = np.quantile(delta_matrix, 0.25, axis=0)
    upper_curve = np.quantile(delta_matrix, 0.75, axis=0)
    color = batch_colors[batch_name]

    axes[0].plot(voltage, median_curve, color=color, label=batch_name)
    axes[0].fill_between(
        voltage, lower_curve, upper_curve, color=color, alpha=0.15
    )

    axes[1].scatter(
        batch_delta['log_delta_q_variance'],
        batch_delta['cycle_life'],
        color=color, alpha=0.7, label=batch_name, edgecolors='white'
    )

axes[0].set_title('Median ΔQ100–10(V) with IQR')
axes[0].set_xlabel('Voltage (V)')
axes[0].set_ylabel('ΔQ (Ah)')
axes[0].legend()

axes[1].set_title('ΔQ Variance vs Cycle Life')
axes[1].set_xlabel('log10(var(ΔQ100–10(V)))')
axes[1].set_ylabel('Cycle Life')
axes[1].legend()

plt.tight_layout()
plt.show()

delta_feature_summary = (
    valid_delta_df.groupby('batch', observed=True)
    .agg(
        cells=('cell_key', 'nunique'),
        log_variance_mean=('log_delta_q_variance', 'mean'),
        log_variance_std=('log_delta_q_variance', 'std'),
        delta_min_mean=('delta_q_min', 'mean'),
        delta_mean=('delta_q_mean', 'mean'),
    )
)
display(delta_feature_summary.round(4))

### 3-1. 장수명 셀과 단수명 셀의 ΔQ(V) 비교

단수명(<500)과 장수명(>1,000) 셀이 모두 존재하는 Batch 2 안에서 비교합니다. 전체 Batch를 합쳐 비교하면 Batch 자체의 분포 차이가 수명 차이처럼 보일 수 있습니다.

In [ ]:
SHORT_LIFE_LIMIT = 500
LONG_LIFE_LIMIT = 1_000
COMPARISON_BATCH = 'Batch 2'
DELTA_Q_GRID_POINTS = 500


def assign_life_group(cycle_life):
    """과제 기준에 따라 셀을 단수명·중간·장수명으로 구분합니다."""
    if cycle_life < SHORT_LIFE_LIMIT:
        return 'Short (<500)'
    if cycle_life > LONG_LIFE_LIMIT:
        return 'Long (>1000)'
    return 'Middle (500–1000)'


def interpolate_delta_q(row, voltage_grid):
    """셀마다 다른 전압 좌표의 ΔQ를 공통 전압 축으로 보간합니다."""
    voltage = np.asarray(row['voltage'], dtype=float)
    delta_q = np.asarray(row['delta_q'], dtype=float)
    order = np.argsort(voltage)

    # np.interp는 오름차순이며 중복되지 않은 x 좌표를 요구합니다.
    unique_voltage, unique_indices = np.unique(voltage[order], return_index=True)
    unique_delta_q = delta_q[order][unique_indices]
    return np.interp(voltage_grid, unique_voltage, unique_delta_q)


delta_feature_df = valid_delta_df.copy()
delta_feature_df['life_group'] = delta_feature_df['cycle_life'].apply(
    assign_life_group
)

# ΔQ 곡선의 크기와 모양을 모델이 사용할 수 있는 스칼라 피처로 요약합니다.
delta_feature_df['delta_q_max'] = delta_feature_df['delta_q'].apply(
    lambda values: float(np.max(values))
)
delta_feature_df['delta_q_range'] = delta_feature_df['delta_q'].apply(
    lambda values: float(np.ptp(values))
)
delta_feature_df['delta_q_abs_mean'] = delta_feature_df['delta_q'].apply(
    lambda values: float(np.mean(np.abs(values)))
)

comparison_df = delta_feature_df[
    (delta_feature_df['batch'] == COMPARISON_BATCH)
    & delta_feature_df['life_group'].isin(['Short (<500)', 'Long (>1000)'])
].copy()

# 모든 비교 셀이 실제로 관측된 공통 전압 구간만 사용합니다.
voltage_start = comparison_df['voltage'].apply(np.min).max()
voltage_end = comparison_df['voltage'].apply(np.max).min()
common_voltage = np.linspace(voltage_start, voltage_end, DELTA_Q_GRID_POINTS)

fig, axis = plt.subplots(figsize=(9, 6))
group_colors = {'Short (<500)': 'tomato', 'Long (>1000)': 'steelblue'}

for group_name in ['Short (<500)', 'Long (>1000)']:
    group_df = comparison_df[comparison_df['life_group'] == group_name]
    curve_matrix = np.vstack([
        interpolate_delta_q(row, common_voltage)
        for _, row in group_df.iterrows()
    ])
    median_curve = np.median(curve_matrix, axis=0)
    lower_curve = np.quantile(curve_matrix, 0.25, axis=0)
    upper_curve = np.quantile(curve_matrix, 0.75, axis=0)
    color = group_colors[group_name]

    axis.plot(
        common_voltage, median_curve, color=color, linewidth=2,
        label=f'{group_name} (n={len(group_df)})',
    )
    axis.fill_between(
        common_voltage, lower_curve, upper_curve, color=color, alpha=0.18
    )

axis.axhline(0, color='black', linewidth=1, linestyle='--')
axis.set_title(f'ΔQ100–10(V): Long vs Short Life — {COMPARISON_BATCH}')
axis.set_xlabel('Voltage (V)')
axis.set_ylabel('ΔQ (Ah)')
axis.legend()
plt.tight_layout()
plt.show()

delta_group_summary = (
    comparison_df.groupby('life_group', observed=True)
    .agg(
        cells=('cell_key', 'nunique'),
        cycle_life_mean=('cycle_life', 'mean'),
        log_variance_mean=('log_delta_q_variance', 'mean'),
        delta_min_mean=('delta_q_min', 'mean'),
        delta_mean=('delta_q_mean', 'mean'),
        delta_abs_mean=('delta_q_abs_mean', 'mean'),
        delta_range_mean=('delta_q_range', 'mean'),
    )
)
display(delta_group_summary.round(4))

### 3-2. ΔQ(V) 통계 피처와 Cycle Life의 관계

선형 관계는 Pearson, 순위 기반 단조 관계는 Spearman 상관계수로 확인합니다. 전체 상관계수는 Batch 분포 차이의 영향을 받을 수 있으므로 Batch별 결과를 함께 봅니다.

In [ ]:
DELTA_Q_FEATURES = [
    'log_delta_q_variance',
    'delta_q_min',
    'delta_q_mean',
    'delta_q_abs_mean',
    'delta_q_range',
]


def build_delta_q_correlation_table(feature_data):
    """전체 및 Batch별 ΔQ 피처–수명 상관계수를 표로 만듭니다."""
    records = []
    datasets = [('Overall', feature_data)]
    datasets.extend(feature_data.groupby('batch', observed=True))

    for dataset_name, dataset in datasets:
        for feature_name in DELTA_Q_FEATURES:
            valid = dataset[[feature_name, 'cycle_life']].dropna()
            records.append({
                'dataset': dataset_name,
                'feature': feature_name,
                'cells': len(valid),
                'pearson_r': valid[feature_name].corr(
                    valid['cycle_life'], method='pearson'
                ),
                'spearman_r': valid[feature_name].corr(
                    valid['cycle_life'], method='spearman'
                ),
            })

    return pd.DataFrame(records)


delta_correlation_df = build_delta_q_correlation_table(delta_feature_df)
correlation_display = delta_correlation_df.assign(
    abs_spearman_r=delta_correlation_df['spearman_r'].abs()
).sort_values(
    ['dataset', 'abs_spearman_r'], ascending=[True, False]
)
display(correlation_display.drop(columns='abs_spearman_r').round(3))

## 4. 충전 조건과 수명의 관계는?

충전 정책별 평균 수명, 표준편차, 표본 수를 배치별로 비교합니다. 표본 수가 작은 정책은 경향 수준으로 해석합니다.

In [ ]:
policy_summary = (
    valid_cell_df.groupby(
        ['batch', 'charging_policy'], observed=True
    )['cycle_life']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(20, 8), sharex=True)

for axis, batch_name in zip(axes, batch_order):
    batch_policy = (
        policy_summary[policy_summary['batch'] == batch_name]
        .sort_values('mean')
    )
    positions = np.arange(len(batch_policy))
    axis.barh(
        positions,
        batch_policy['mean'],
        xerr=batch_policy['std'].fillna(0),
        color='steelblue',
        alpha=0.75,
    )
    axis.set_yticks(positions)
    axis.set_yticklabels(batch_policy['charging_policy'], fontsize=8)
    axis.set_title(batch_name)
    axis.set_xlabel('Mean Cycle Life')

    for position, (_, row) in enumerate(batch_policy.iterrows()):
        axis.text(row['mean'], position, f" n={int(row['count'])}",
                  va='center', fontsize=7)

fig.suptitle('Cycle Life by Charging Policy', fontweight='bold')
plt.tight_layout()
plt.show()

display(policy_summary.round(2))

### 4-1. C-rate와 수명·열화 속도의 관계

`C1(전환 비율)-C2` 형식의 충전 프로토콜에서 두 C-rate와 전환 비율을 분리합니다. `rate_exposure_proxy`는 80% SOC까지 각 전류가 적용되는 용량 구간으로 가중한 비교용 지표이며, 실제 시간 평균 전류는 아닙니다.

In [ ]:
POLICY_PATTERN = (
    r'(?P<first_c_rate>\d+(?:\.\d+)?)C'
    r'\((?P<switch_pct>\d+)%\)-'
    r'(?P<second_c_rate>\d+(?:\.\d+)?)C'
)
CHARGE_RANGE_END_PCT = 80


def extract_charging_features(cell_data):
    """충전 프로토콜 문자열을 모델링 가능한 수치 피처로 변환합니다."""
    result = cell_data.copy()
    extracted = result['charging_policy'].str.extract(POLICY_PATTERN)
    numeric_columns = ['first_c_rate', 'switch_pct', 'second_c_rate']
    extracted[numeric_columns] = extracted[numeric_columns].astype(float)
    result[numeric_columns] = extracted[numeric_columns]
    # newstructure는 원본 충전 알고리즘 문자열에 포함된 메타데이터 태그입니다.
    # 셀 소재나 물리적 구조가 바뀌었다는 뜻으로 해석하지 않습니다.
    result['has_newstructure_tag'] = result['charging_policy'].str.contains(
        'newstructure', case=False, na=False
    )

    # 첫 번째·두 번째 전류가 적용되는 용량 구간을 반영한 노출 지표입니다.
    result['rate_exposure_proxy'] = (
        result['first_c_rate'] * result['switch_pct']
        + result['second_c_rate']
        * (CHARGE_RANGE_END_PCT - result['switch_pct'])
    ) / CHARGE_RANGE_END_PCT
    return result


def build_rate_correlation_table(data, outcome_columns):
    """전체 및 Batch별 C-rate 피처의 Spearman 상관계수를 계산합니다."""
    predictors = [
        'first_c_rate', 'switch_pct',
        'second_c_rate', 'rate_exposure_proxy',
    ]
    records = []
    datasets = [('Overall', data)]
    datasets.extend(data.groupby('batch', observed=True))

    for dataset_name, dataset in datasets:
        for outcome_name in outcome_columns:
            for predictor_name in predictors:
                valid = dataset[[predictor_name, outcome_name]].dropna()
                records.append({
                    'dataset': dataset_name,
                    'outcome': outcome_name,
                    'charging_feature': predictor_name,
                    'cells': len(valid),
                    'spearman_r': valid[predictor_name].corr(
                        valid[outcome_name], method='spearman'
                    ),
                })

    return pd.DataFrame(records)


charging_feature_df = extract_charging_features(valid_cell_df)
life_rate_correlation = build_rate_correlation_table(
    charging_feature_df, ['cycle_life']
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for axis, batch_name in zip(axes, batch_order):
    batch_data = charging_feature_df[
        charging_feature_df['batch'] == batch_name
    ]
    for is_new, marker, label in [
        (False, 'o', 'No tag'), (True, '^', 'newstructure tag')
    ]:
        subset = batch_data[batch_data['has_newstructure_tag'] == is_new]
        if subset.empty:
            continue
        axis.scatter(
            subset['rate_exposure_proxy'], subset['cycle_life'],
            marker=marker, alpha=0.75, label=label,
        )
    axis.set_title(batch_name)
    axis.set_xlabel('C-rate Exposure Proxy')
    axis.legend(fontsize=8)
axes[0].set_ylabel('Cycle Life')
fig.suptitle('Charging-rate Exposure vs Cycle Life', fontweight='bold')
plt.tight_layout()
plt.show()

display(life_rate_correlation.round(3))

In [ ]:
charging_degradation_df = knee_df.merge(
    charging_feature_df[[
        'batch', 'cell_key', 'first_c_rate', 'switch_pct',
        'second_c_rate', 'rate_exposure_proxy', 'has_newstructure_tag',
    ]],
    on=['batch', 'cell_key'],
    how='inner',
)

# 기존 기울기는 용량 감소 때문에 음수이므로, 해석하기 쉬운 양의 열화 속도로 변환합니다.
charging_degradation_df['before_degradation_speed'] = (
    -charging_degradation_df['before_slope_mAh_per_cycle']
)
charging_degradation_df['after_degradation_speed'] = (
    -charging_degradation_df['after_slope_mAh_per_cycle']
)

degradation_rate_correlation = build_rate_correlation_table(
    charging_degradation_df,
    ['before_degradation_speed', 'after_degradation_speed'],
)
display(degradation_rate_correlation.round(3))

## 5. 초기 신호 중 무엇이 수명과 연관되어 있는가?

각 셀의 초기 100사이클만 사용해 피처를 생성하고, 배치별 상관관계를 같은 축으로 비교합니다.

In [ ]:
def build_early_features(summary_data, delta_data, max_cycle=100):
    """셀별 초기 max_cycle 이내의 모델 후보 피처를 생성합니다."""
    early = summary_data[summary_data['cycle'] <= max_cycle]
    records = []

    for (batch_name, cell_key), cell_data in early.groupby(
        ['batch', 'cell_key'], observed=True
    ):
        records.append({
            'batch': batch_name,
            'cell_key': cell_key,
            'cycle_life': cell_data['cycle_life'].iloc[0],
            'mean_QD': cell_data['QDischarge'].mean(),
            'std_QD': cell_data['QDischarge'].std(),
            'mean_IR': cell_data['IR'].mean(),
            'mean_Tavg': cell_data['Tavg'].mean(),
            'mean_Tmax': cell_data['Tmax'].mean(),
            'mean_chargetime': cell_data['chargetime'].mean(),
        })

    features = pd.DataFrame(records)
    delta_features = delta_data.drop(
        columns=['voltage', 'delta_q', 'cycle_life'], errors='ignore'
    )
    return features.merge(
        delta_features, on=['batch', 'cell_key'], how='left'
    )


early_feature_df = build_early_features(valid_cycle_df, valid_delta_df)
display(early_feature_df.head())

In [ ]:
correlation_columns = [
    'cycle_life',
    'mean_QD',
    'std_QD',
    'mean_IR',
    'mean_Tavg',
    'mean_Tmax',
    'mean_chargetime',
    'log_delta_q_variance',
]

fig, axes = plt.subplots(1, 3, figsize=(22, 7), constrained_layout=True)
ranking_records = []

for axis, batch_name in zip(axes, batch_order):
    batch_features = early_feature_df[early_feature_df['batch'] == batch_name]
    correlation = batch_features[correlation_columns].corr()
    image = axis.imshow(correlation, cmap='RdBu_r', vmin=-1, vmax=1)
    labels = correlation.columns.tolist()
    axis.set_xticks(range(len(labels)))
    axis.set_yticks(range(len(labels)))
    axis.set_xticklabels(labels, rotation=55, ha='right', fontsize=8)
    axis.set_yticklabels(labels, fontsize=8)
    axis.set_title(batch_name)

    for row in range(len(labels)):
        for column in range(len(labels)):
            value = correlation.iloc[row, column]
            text_color = 'white' if abs(value) > 0.5 else 'black'
            axis.text(column, row, f'{value:.2f}', ha='center', va='center',
                      fontsize=7, color=text_color)

    target_correlations = correlation['cycle_life'].drop('cycle_life')
    for feature, value in target_correlations.items():
        ranking_records.append({
            'batch': batch_name,
            'feature': feature,
            'correlation_with_cycle_life': value,
            'absolute_correlation': abs(value),
        })

fig.colorbar(image, ax=axes, shrink=0.75, label='Pearson correlation')
fig.suptitle('Early 100-Cycle Feature Correlations', fontweight='bold')
plt.show()

correlation_ranking = (
    pd.DataFrame(ranking_records)
    .sort_values(['batch', 'absolute_correlation'], ascending=[True, False])
)
display(correlation_ranking.round(3))

### 5-1. 다중공선성 확인

피처끼리 절대 Pearson 상관계수가 0.8 이상인 조합을 찾고 VIF를 계산합니다. 일반적으로 VIF 5 이상은 주의, 10 이상은 강한 다중공선성 신호로 해석합니다.

In [ ]:
MODEL_FEATURES = [
    column for column in correlation_columns if column != 'cycle_life'
]
HIGH_CORRELATION_LIMIT = 0.8


def find_high_correlation_pairs(data, feature_columns, threshold=0.8):
    """중복 정보 가능성이 큰 피처 조합을 반환합니다."""
    correlation = data[feature_columns].corr()
    records = []

    for left_index, left_feature in enumerate(feature_columns):
        for right_feature in feature_columns[left_index + 1:]:
            value = correlation.loc[left_feature, right_feature]
            if pd.notna(value) and abs(value) >= threshold:
                records.append({
                    'feature_1': left_feature,
                    'feature_2': right_feature,
                    'correlation': value,
                    'absolute_correlation': abs(value),
                })

    return records


def calculate_vif(data, feature_columns):
    """외부 라이브러리 없이 각 피처의 VIF를 계산합니다."""
    clean = data[feature_columns].replace([np.inf, -np.inf], np.nan).dropna()
    standardized = (clean - clean.mean()) / clean.std(ddof=0)
    matrix = standardized.to_numpy(dtype=float)
    records = []

    for feature_index, feature_name in enumerate(feature_columns):
        target = matrix[:, feature_index]
        predictors = np.delete(matrix, feature_index, axis=1)
        predictors = np.column_stack([np.ones(len(predictors)), predictors])
        coefficients = np.linalg.lstsq(predictors, target, rcond=None)[0]
        predictions = predictors @ coefficients
        residual_sum = np.square(target - predictions).sum()
        total_sum = np.square(target - target.mean()).sum()
        r_squared = 1 - residual_sum / total_sum
        vif = np.inf if 1 - r_squared <= 1e-12 else 1 / (1 - r_squared)
        records.append({
            'feature': feature_name,
            'cells': len(clean),
            'VIF': vif,
        })

    return records


high_correlation_records = []
vif_records = []
for batch_name in batch_order:
    batch_features = early_feature_df[early_feature_df['batch'] == batch_name]

    for record in find_high_correlation_pairs(
        batch_features, MODEL_FEATURES, HIGH_CORRELATION_LIMIT
    ):
        high_correlation_records.append({'batch': batch_name, **record})

    for record in calculate_vif(batch_features, MODEL_FEATURES):
        vif_records.append({'batch': batch_name, **record})

high_correlation_df = pd.DataFrame(high_correlation_records).sort_values(
    ['batch', 'absolute_correlation'], ascending=[True, False]
)
vif_df = pd.DataFrame(vif_records).sort_values(
    ['batch', 'VIF'], ascending=[True, False]
)

print(f'|상관계수| ≥ {HIGH_CORRELATION_LIMIT}인 피처 조합')
display(high_correlation_df.round(3))
print('Batch별 VIF')
display(vif_df.round(2))

## 6. 모델 설계 전략

### 문제 정의
- 유형: Regression
- 관측 단위: 배터리 셀 1개당 1행
- 입력: 초기 100사이클에서 생성한 피처
- Target: `cycle_life`

### 최종 입력 피처
- `log_delta_q_variance`: 초기 열화 신호
- `mean_chargetime`: 충전 속도 및 조건 신호
- 두 피처의 단위가 다르므로 `StandardScaler` 적용
- `mean_Tmax`, Knee point, 셀·Batch 식별자는 입력에서 제외

### 학습 및 평가
- Batch 1: 모델 학습과 반복 교차검증
- Batch 2: 1차 테스트
- Batch 3: 2차 테스트
- 전처리는 `Pipeline` 내부에서 각 학습 fold에만 `fit`
- 모든 무작위 단계에 `RANDOM_STATE` 적용

### 후보 모델 및 최종 선택
1. DummyRegressor: 평균 예측 기준선
2. LinearRegression/Ridge: 규제 효과를 비교하기 위한 선형 기준 모델
3. ElasticNet: 작은 표본에서 계수 크기를 제한하는 최종 1순위 모델
4. SVR: 비선형 관계가 성능을 개선하는지 확인하는 비교 모델
- Random Forest·Gradient Boosting은 학습 Target 범위 밖으로 외삽하기 어려워 우선순위를 낮춤
- 최종 모델은 Batch 1 교차검증 결과로 확정하며, Batch 2·3으로 모델을 재선택하지 않음

### 평가 지표
- MAE: 평균적인 사이클 오차
- RMSE: 큰 오차에 더 큰 패널티
- MAPE: 수명 대비 상대 오차
- R²: 설명력

### 누수 방지
- 전체 수명으로 계산한 Knee point는 모델 입력에서 제외
- 같은 셀의 여러 사이클을 서로 다른 fold로 분할하지 않음
- Batch 2·3의 테스트 성능을 본 뒤 모델을 재선택하지 않음